In [1]:
# NEEDED FOR RESAMPLING USING TORCHAUDIO
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# import torch

# torch.set_num_threads(1)
# torch.set_num_interop_threads(1)

In [2]:
import sys
from tqdm import tqdm

BASE_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr"
VOCAB_FILE = f"{BASE_PATH}/src/model/xeusphoneme/resources/ipa_vocab.json"

sys.path.append(BASE_PATH)

# ipapack
from src.data.kaldi_pretraining_dataset import build_kaldi_datamodule

datamodule = build_kaldi_datamodule(
    train_split="dev_1k",  # "train_accentmix_multi",
    dev_splits=[
        # "dev_1k",
        # "dev_gmuaccent",
        # "dev_buckeye",
        "dev_epadb",
        # "dev_speechoceanotth",
        # "dev_l2arctic",
    ],
    predict_split="predict",
    dataset_config_path=f"{BASE_PATH}/configs/data/ipapack_index.yaml",
    batch_size=1,
    num_workers=1,
    vocab_file=VOCAB_FILE,
    # limit_samples=2,
)

# from src.data.kaldi_dataset import build_kaldi_datamodule

# DATASET = "buckeye"
# datamodule = build_kaldi_datamodule(
#     DATASET,
#     data_dir="/work/hdd/bbjs/shared/powsm/s2t1/dump/raw",
#     dataset_config_path=f"{BASE_PATH}/configs/data/powsm_evalset_index.yaml",
#     portable_wavscp=False,
#     sampling_rate=16000,
#     batch_size=1,
#     num_workers=1,
#     vocab_file=VOCAB_FILE,
# )

datamodule.setup()
dataloader = datamodule.test_dataloader()

print("Loaded dataset with length:", len(dataloader))

/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/.venv_dai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 4655 samples from wav.scp: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/dev_1k_fixed/wav.scp with count 0


Reading text: 18620it [00:00, 180060.36it/s]
Reading language: 18620it [00:00, 156688.51it/s]


Loaded 150 samples from wav.scp: /work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/data/devsets/epadb.scp with count 0


Reading text: 3159it [00:00, 57866.74it/s]
Reading language: 3159it [00:00, 66411.07it/s]


Loaded 4655 samples from wav.scp: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/dev_1k_fixed/wav.scp with count 0


Reading text: 18620it [00:00, 581323.76it/s]
Reading language: 18620it [00:00, 586637.98it/s]

Loaded dataset with length: 4655


In [5]:
import torch
from src.model.wav2vec2.builders import build_wav2vec2pr_inference
import json

CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/ipaaccent_ctc/mms_multiaccent.bs256.lr1em5/checkpoints/checkpoint-500.ckpt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device = ", device)

inference = build_wav2vec2pr_inference(
    hf_repo="facebook/mms-300m",
    vocab_file=VOCAB_FILE,
    checkpoint=CKPT_PATH,
)

with open(VOCAB_FILE, "r") as f:
    vocab = json.load(f)
id2token = {k: v for v, k in vocab.items()}

Using device =  cuda


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sampling rate: 16000
Loaded checkpoint: /work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/ipaaccent_ctc/mms_multiaccent.bs256.lr1em5/checkpoints/checkpoint-500.ckpt with load info: <All keys matched successfully>


In [ ]:
from src.metrics.phone_recognition import PhoneRecognitionEvaluator

evaluator = PhoneRecognitionEvaluator()


def get_phone_str(token_ids):
    return "/".join([id2token[t] for t in token_ids if t in id2token])


N_SAMPLES = 150
n_data = len(dataloader)
print("Total data samples:", n_data, "sampling ", N_SAMPLES)
results = []
with torch.no_grad():
    for bidx, batch in tqdm(enumerate(dataloader), desc="Making predictions"):
        # if bidx % (n_data // min(N_SAMPLES, n_data)) != 0:
        #     continue
        # assert batch size is 1
        print(batch)
        batch = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }

        prediction = inference(**batch)
        for i, key in enumerate(batch["keys"]):
            pred = prediction[i]["processed_transcript"]
            gt_str = batch["text"][i]
            if isinstance(gt_str, torch.Tensor):
                gt_str = gt_str.cpu().numpy()
                gt_phones = get_phone_str(gt_str)
            else:
                gt_phones = gt_str
            prmetrics, _ = evaluator.evaluate(
                {i: {"prediction": pred, "transcription": gt_phones.replace("/", "")}},
                compute_inventory=False,
            )
            asr_text = batch["asr_text"][i] if "asr_text" in batch else ""
            results.append(
                {
                    "key": key,
                    "speech": batch["speech"][i].cpu().numpy(),
                    "speech_length": batch["speech_length"][i].cpu().item(),
                    "wavpath": batch["wavpath"][i],
                    "phone_str": gt_phones,
                    "language": batch["lang_sym"][i],
                    "asr_text": asr_text,
                    "prediction": prediction[i]["predicted_transcript"],
                    "pr_metrics": prmetrics,
                }
            )
        if bidx > 1:
            break
print(f"Collected {len(results)} samples for error analysis.")

Total data samples: 4655 sampling  150


Making predictions: 0it [00:00, ?it/s]

{'keys': ['aaaaa_cv_dev_0000000000000000008461769_pr'], 'speech': tensor([[0., 0., 0.,  ..., 0., 0., 0.]]), 'speech_length': tensor([94464]), 'text': tensor([[212, 158, 189, 190, 181,  43,  90, 360, 185,  88, 366, 360,  42, 360,
         340,  43,  24, 288, 408,  24, 365, 360,  17, 227,  95, 367, 360, 153,
         181, 415, 340, 117, 201, 181, 278, 340, 367, 285, 330, 190, 181,  95,
         415,  43,  17, 117, 366, 189, 415, 367, 117, 415, 360, 142, 360, 103,
         415, 288, 185, 340,  24, 413,  24, 360, 103, 330, 415, 190, 360, 158,
         189,  17, 408,  24]]), 'text_length': tensor([74]), 'wavpath': ['/work/hdd/bbjs/shared/powsm/s2t1/dump/raw/org/dev/data/format.10/data_wav.ark:205380909'], 'lang_sym': ['rus'], 'asr_text': [None]}
tensor([[[ 3.0251, -0.1360, -0.2660,  ..., -0.2681, -0.1772, -0.1381],
         [ 3.0507, -0.1071, -0.2668,  ..., -0.2632, -0.1756, -0.1415],
         [ 3.0844, -0.1061, -0.2689,  ..., -0.2541, -0.1796, -0.1489],
         ...,
         [ 3.0033, -0.

Making predictions: 1it [00:14, 14.76s/it]

{'keys': ['aaaaa_cv_dev_0000000000000000008458168_pr'], 'speech': tensor([[0., 0., 0.,  ..., 0., 0., 0.]]), 'speech_length': tensor([105408]), 'text': tensor([[ 17, 227, 288, 185, 103,  42, 189, 408, 181, 307, 123, 278,  51, 408,
          51,  17, 227, 340, 181, 408, 181, 307, 123, 278, 181, 408, 181, 415,
         181, 340,  51, 310, 189, 375, 181,  17, 303, 181, 132, 189, 269, 288,
         189, 340]]), 'text_length': tensor([44]), 'wavpath': ['/work/hdd/bbjs/shared/powsm/s2t1/dump/raw/org/dev/data/format.9/data_wav.ark:789624994'], 'lang_sym': ['kmr'], 'asr_text': [None]}
tensor([[[ 3.0352, -0.1322, -0.2611,  ..., -0.2657, -0.1748, -0.1451],
         [ 3.0290, -0.1189, -0.2617,  ..., -0.2620, -0.1717, -0.1489],
         [ 3.0125, -0.1144, -0.2620,  ..., -0.2585, -0.1718, -0.1429],
         ...,
         [ 3.0066, -0.1233, -0.2624,  ..., -0.2584, -0.1748, -0.1408],
         [ 3.0108, -0.1249, -0.2611,  ..., -0.2582, -0.1770, -0.1403],
         [ 3.0188, -0.1327, -0.2533,  ..., -0.25

Making predictions: 2it [00:28, 13.92s/it]

{'keys': ['aaaaa_cv_dev_0000000000000000008616073_pr'], 'speech': tensor([[0., 0., 0.,  ..., 0., 0., 0.]]), 'speech_length': tensor([61056]), 'text': tensor([[375, 181, 240, 240, 181, 187, 415,  40, 340, 398, 310, 398, 183, 375,
         189, 103, 415, 189, 240, 375, 183, 101, 132, 398]]), 'text_length': tensor([24]), 'wavpath': ['/work/hdd/bbjs/shared/powsm/s2t1/dump/raw/org/dev/data/format.23/data_wav.ark:195876837'], 'lang_sym': ['bak'], 'asr_text': [None]}
tensor([[[ 3.0013, -0.1344, -0.2600,  ..., -0.2551, -0.1830, -0.1419],
         [ 2.9918, -0.1230, -0.2610,  ..., -0.2519, -0.1780, -0.1421],
         [ 2.9842, -0.1223, -0.2607,  ..., -0.2517, -0.1784, -0.1378],
         ...,
         [ 2.9895, -0.1253, -0.2621,  ..., -0.2518, -0.1761, -0.1378],
         [ 2.9930, -0.1266, -0.2605,  ..., -0.2515, -0.1784, -0.1370],
         [ 3.0013, -0.1332, -0.2507,  ..., -0.2448, -0.1858, -0.1416]]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 

Making predictions: 2it [00:41, 20.71s/it]

Collected 3 samples for error analysis.


In [7]:
results

[{'key': 'aaaaa_cv_dev_0000000000000000008461769_pr',
  'speech': array([0., 0., 0., ..., 0., 0., 0.], shape=(320000,), dtype=float32),
  'speech_length': 94464,
  'wavpath': '/work/hdd/bbjs/shared/powsm/s2t1/dump/raw/org/dev/data/format.10/data_wav.ark:205380909',
  'phone_str': 'vʲ/mʲ/i/rʲ/e/f/sʲ/ɪ/o/p/ɕ/ɪ/j/ɪ/n/f/ə/r/m/ə/tʲ/ɪ/z/a/t͡s/ɨ/ɪ/nʲ/e/t/n/ɐ/dʲ/e/ʐ/n/ɨ/x/s/rʲ/e/t͡s/t/f/z/ɐ/ɕ/i/t/ɨ/ɐ/t/ɪ/lʲ/ɪ/k/t/r/o/n/ə/v/ə/ɪ/k/s/t/rʲ/ɪ/mʲ/i/z/m/ə',
  'language': 'rus',
  'asr_text': None,
  'prediction': '',
  'pr_metrics': PhoneRecognitionSummary(PFER=74.0, FER=92.1734234234234, FED=68.20833333333331, PER=100.0, SUB=0.0, INS=0.0, DEL=100.0, N=1, phones=74, inventory=None)},
 {'key': 'aaaaa_cv_dev_0000000000000000008458168_pr',
  'speech': array([0., 0., 0., ..., 0., 0., 0.], shape=(320000,), dtype=float32),
  'speech_length': 105408,
  'wavpath': '/work/hdd/bbjs/shared/powsm/s2t1/dump/raw/org/dev/data/format.9/data_wav.ark:789624994',
  'phone_str': 'z/a/r/o/k/j/i/m/e/h/ɑ/ʐ/ɛ/m/ɛ/z/a/n/e/m/

In [ ]:
sp = batch["speech"].cpu().numpy()
from IPython.display import Audio

Audio(sp, rate=16000)